# 00 — Démarrage d'une session Colab

À exécuter **au début de chaque session**, avant tout autre notebook.

Ce notebook :
1. monte Google Drive (données, vecteurs, résultats) ;
2. récupère le code depuis GitHub (dépôt public) ;
3. installe le paquet `facegeom` ;
4. affiche les versions et le GPU ;
5. si besoin, copie et décompresse CelebA sur la machine Colab.

**GPU** : Exécution → Modifier le type d'exécution → GPU T4.

## 1. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Dossier du projet sur Drive (compte d'Arsène, partagé avec le groupe)
DRIVE_ROOT = Path("/content/drive/MyDrive/srd-facegeom")
for sub in ["data", "features", "runs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive :", DRIVE_ROOT, sorted(p.name for p in DRIVE_ROOT.iterdir()))

## 2. Code depuis GitHub

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Arsenicophe/srd-facegeom.git"
BRANCH = "main"
CODE_DIR = Path("/content/srd-facegeom")

if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(CODE_DIR)], check=True)

os.chdir(CODE_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Installation du paquet

In [ ]:
%pip install -q -e "/content/srd-facegeom[dev]"

import importlib, facegeom
importlib.reload(facegeom)
print("facegeom", facegeom.__version__)

## 4. Versions et GPU

In [ ]:
import sys, torch, torchvision
import pytorch_metric_learning as pml

print("Python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("pml        ", pml.__version__)
print("CUDA       ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— pas de GPU")

## 5. CelebA sur la machine Colab (si besoin)

Seulement pour les étapes qui lisent les images (extraction, méthodes classiques).
Le zip est sur Drive ; le décompresser directement sur Drive serait très lent,
donc on le copie sur le disque local de Colab.

In [ ]:
BESOIN_IMAGES = False  # mettre True pour les étapes qui lisent les images

import shutil, zipfile
LOCAL_DATA = Path("/content/data")
CELEBA_ZIP = DRIVE_ROOT / "data" / "img_align_celeba.zip"

if BESOIN_IMAGES:
    if not CELEBA_ZIP.exists():
        raise FileNotFoundError(f"{CELEBA_ZIP} absent : voir l'étape 0e")
    LOCAL_DATA.mkdir(exist_ok=True)
    if not (LOCAL_DATA / "img_align_celeba").exists():
        shutil.copy(CELEBA_ZIP, LOCAL_DATA / CELEBA_ZIP.name)
        with zipfile.ZipFile(LOCAL_DATA / CELEBA_ZIP.name) as z:
            z.extractall(LOCAL_DATA)
        (LOCAL_DATA / CELEBA_ZIP.name).unlink()
    print(len(list((LOCAL_DATA / "img_align_celeba").glob("*.jpg"))), "images")
else:
    print("Images non copiées (BESOIN_IMAGES = False)")